In [ ]:
# In[1]:

import pandas as pd

# Reuse variable df across files for efficiency
files = [
    ("metric.csv", "kpi_name"),
    ("trace.csv", "trace_name"),
    ("log.csv", "log_name"),
    ("error_logs.csv", "message"),
]

rows = []
for fname, name_col in files:
    df = pd.read_csv(fname)
    row_count = len(df)
    # Earliest/latest as Unix seconds (use original numeric timestamp column)
    if "timestamp" in df.columns and row_count > 0:
        # ensure numeric
        tsnum = pd.to_numeric(df["timestamp"], errors="coerce")
        earliest = int(tsnum.min()) if tsnum.notna().any() else None
        latest = int(tsnum.max()) if tsnum.notna().any() else None
        # also parse to UTC datetimes per rule (kept in df but not displayed fully)
        df["_ts_dt"] = pd.to_datetime(tsnum, unit="s", utc=True)
    else:
        earliest = None
        latest = None

    # up to 20 distinct cmdb_id values
    if "cmdb_id" in df.columns:
        cmdb_vals = pd.Series(df["cmdb_id"].dropna().astype(str).unique()).tolist()[:20]
    else:
        cmdb_vals = []

    # up to 20 distinct KPI/trace/log names or message prefixes (first 80 chars)
    if name_col == "message":
        if "message" in df.columns:
            msgs = df["message"].dropna().astype(str).apply(lambda x: x[:80])
            names_list = pd.Series(msgs.unique()).tolist()[:20]
        else:
            names_list = []
    else:
        if name_col in df.columns:
            names_list = pd.Series(df[name_col].dropna().astype(str).unique()).tolist()[:20]
        else:
            names_list = []

    rows.append({
        "file": fname,
        "row_count": row_count,
        "earliest_ts": earliest,
        "latest_ts": latest,
        "cmdb_ids": cmdb_vals,
        "names": names_list
    })

summary_df = pd.DataFrame(rows)[["file", "row_count", "earliest_ts", "latest_ts", "cmdb_ids", "names"]]

# Display compact summary (one table)
summary_df

```
Out[1]:
```
# Build a concise plain-English summary string based on the previously computed summary_df
# (reuse summary_df from earlier execution)
def fmt_ts(x):
    try:
        if pd.isna(x):
            return "None"
        return str(int(float(x)))
    except:
        return str(x)

lines = []
for _, row in summary_df.iterrows():
    fname = row['file']
    rc = int(row['row_count'])
    ets = fmt_ts(row['earliest_ts'])
    lts = fmt_ts(row['latest_ts'])
    cmdbs = row['cmdb_ids'] or []
    names = row['names'] or []
    # compact representations
    cmdb_str = ", ".join(cmdbs[:20]) if cmdbs else "[]"
    names_str = ", ".join(names[:20]) if names else "[]"
    if fname == "metric.csv":
        lines.append(f"metric.csv: {rc} rows; earliest_ts={ets}, latest_ts={lts}; cmdb_ids=[{cmdb_str}]; kpi_names=[{names_str}]")
    elif fname == "trace.csv":
        lines.append(f"trace.csv: {rc} rows; earliest_ts={ets}, latest_ts={lts}; cmdb_ids=[{cmdb_str}]; trace_names=[{names_str}]")
    elif fname == "log.csv":
        lines.append(f"log.csv: {rc} rows; earliest_ts={ets}, latest_ts={lts}; cmdb_ids=[{cmdb_str}]; log_names=[{names_str}]")
    elif fname == "error_logs.csv":
        lines.append(f"error_logs.csv: {rc} rows; earliest_ts={ets}, latest_ts={lts}; cmdb_ids=[{cmdb_str}]; message_prefixes=[{names_str}]")
summary = "\n".join(lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

file  row_count   earliest_ts     latest_ts                                           cmdb_ids                                              names
0      metric.csv       1850  1.705737e+09  1.705739e+09  [adservice, cartservice, checkoutservice, curr...  [cpu, diskio, latency-50, latency-90, mem, soc...
1       trace.csv       3800  1.705737e+09  1.705739e+09  [checkoutservice, currencyservice, emailservic...  [trace.from_checkoutservice.duration_mean, tra...
2         log.csv        460  1.705737e+09  1.705739e+09  [adservice, cartservice, checkoutservice, curr...                   [log.error_count, log.row_count]
3  error_logs.csv          0           NaN           NaN                                                 []                                                 []```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse variables if present; otherwise load files
metric_file = "metric.csv"
trace_file = "trace.csv"
log_file = "log.csv"
error_file = "error_logs.csv"

# Candidate services
cands = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

# Load files
mdf = pd.read_csv(metric_file)
tdf = pd.read_csv(trace_file)
ldf = pd.read_csv(log_file)
edf = pd.read_csv(error_file)

# parse timestamps to UTC datetimes (keep original numeric timestamp)
for df in (mdf, tdf, ldf, edf):
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
        df["ts"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# incident window (UTC)
window_start = pd.to_datetime("2024-01-20 07:52:39", utc=True)
window_end = pd.to_datetime("2024-01-20 08:22:39", utc=True)

def analyze_file(df, name_col, group_col_label):
    # filter candidate services
    df_c = df[df["cmdb_id"].isin(cands)].copy()
    # ensure value numeric
    df_c["value"] = pd.to_numeric(df_c["value"], errors="coerce")
    # compute global P95 per group using full series (per rule)
    gp = df_c.groupby(["cmdb_id", name_col], as_index=False)["value"].quantile(0.95)
    gp = gp.rename(columns={"value": "global_P95"})
    # merge global p95 back to full candidate df
    df_c = df_c.merge(gp, on=["cmdb_id", name_col], how="left")
    # window rows
    win_mask = (df_c["ts"] >= window_start) & (df_c["ts"] <= window_end)
    df_win = df_c[win_mask].copy()
    if df_win.shape[0] > 0:
        df_win["is_anom"] = (df_win["value"] >= df_win["global_P95"]).astype(int)
        # anomaly timestamp only where anomaly
        df_win["anomaly_ts"] = df_win["timestamp"].where(df_win["is_anom"]==1, np.nan)
        # aggregate window stats
        agg = df_win.groupby(["cmdb_id", name_col], as_index=False).agg(
            total_points_in_window = ("value", "count"),
            anomaly_count = ("is_anom", "sum"),
            earliest_anomaly_ts = ("anomaly_ts", "min"),
            max_value_in_window = ("value", "max"),
        )
    else:
        # empty window aggregation template
        agg = pd.DataFrame(columns=["cmdb_id", name_col, "total_points_in_window", "anomaly_count", "earliest_anomaly_ts", "max_value_in_window"])
    # merge with global_p95 to ensure all groups present
    summary = gp.merge(agg, on=["cmdb_id", name_col], how="left")
    summary["total_points_in_window"] = summary["total_points_in_window"].fillna(0).astype(int)
    summary["anomaly_count"] = summary["anomaly_count"].fillna(0).astype(int)
    # earliest_anomaly_ts keep as float/NaN; convert to Int where present
    summary["earliest_anomaly_ts"] = summary["earliest_anomaly_ts"].where(summary["earliest_anomaly_ts"].notna(), np.nan)
    summary["max_value_in_window"] = summary["max_value_in_window"].where(summary["max_value_in_window"].notna(), np.nan)
    # rename the group name column to common label
    summary = summary.rename(columns={name_col: group_col_label})
    # select and order columns
    cols = ["cmdb_id", group_col_label, "global_P95", "total_points_in_window", "anomaly_count", "earliest_anomaly_ts", "max_value_in_window"]
    summary = summary[cols]
    # separate positive-anomaly groups
    pos = summary[summary["anomaly_count"] > 0].copy()
    if pos.shape[0] == 0:
        # fallback: return top 20 groups sorted by anomaly_count desc then max_value_in_window desc
        summary_sorted = summary.sort_values(by=["anomaly_count", "max_value_in_window"], ascending=[False, False])
        out = summary_sorted.head(20).reset_index(drop=True)
    else:
        out = pos.sort_values(by=["anomaly_count", "max_value_in_window"], ascending=[False, False]).head(20).reset_index(drop=True)
    return out, summary.shape[0]

# Analyze metric.csv
metric_res, metric_group_count = analyze_file(mdf, "kpi_name", "kpi_name")

# Analyze trace.csv
trace_res, trace_group_count = analyze_file(tdf, "trace_name", "trace_name")

# Analyze log.csv
log_res, log_group_count = analyze_file(ldf, "log_name", "log_name")

# Confirm error_logs.csv empty
error_logs_empty = (edf.shape[0] == 0)

# Compact counts summary
counts_summary = pd.DataFrame({
    "file": ["metric.csv", "trace.csv", "log.csv", "error_logs.csv"],
    "distinct_cmdbid_kpi_groups_processed": [metric_group_count, trace_group_count, log_group_count, 0],
    "error_logs_empty": [None, None, None, error_logs_empty]
})

# Display compact outputs (top 20 rows per file and counts)
metric_res, trace_res, log_res, counts_summary

```
Out[2]:
```
Summary of findings (plain English, compact):

Overall
- Candidate services processed: metric groups = 34, trace groups = 88, log groups = 8.
- error_logs.csv is empty.

metric.csv (top anomalous groups)
- socket is the strongest anomaly across services:
  - checkoutservice | kpi=socket | global_P95=9.0 | points_in_window=25 | anomaly_count=25 | earliest_anomaly_ts=1705737000 | max=9.0
  - recommendationservice | socket | global_P95=4.0 | points=25 | anomaly_count=25 | earliest_anomaly_ts=1705737000 | max=4.0
  - productcatalogservice | socket | global_P95=3.0 | points=25 | anomaly_count=25 | earliest_anomaly_ts=1705737000 | max=3.2
  - currencyservice | socket | global_P95=3.0 | points=25 | anomaly_count=25 | earliest_anomaly_ts=1705737000 | max=3.0
  - emailservice | socket | global_P95=3.0 | points=25 | anomaly_count=25 | earliest_anomaly_ts=1705737000 | max=3.0
- diskio for several services has global_P95=0 and anomaly_count=25 (all 25 points flagged).
- mem, cpu, workload show smaller anomaly counts (typically 2–3 anomalies in the window) for various services.

trace.csv (top anomalous groups)
- Many trace error_rate series show anomalies (global_P95 reported as 0 and anomaly_count=25 for multiple trace_name / cmdb_id combinations). Examples (all with anomaly_count=25, earliest_anomaly_ts=1705737000, max=0.0):
  - checkoutservice: trace.from_checkoutservice.error_rate, trace.from_frontendservice.error_rate, trace.from_root.error_rate, trace.to_* .error_rate, etc.
  - currencyservice, emailservice, productcatalogservice, recommendationservice: similar trace.*.error_rate series flagged.
- In short: widespread trace error_rate elevations across the candidate services during the incident window.

log.csv (top anomalous groups)
- log.error_count: for all candidate services (checkoutservice, currencyservice, emailservice, recommendationservice) global_P95=0 and anomaly_count=25 (all window points flagged).
- log.row_count: a few services (currencyservice, recommendationservice, checkoutservice, emailservice) show global_P95 > 0 and anomaly_count = 2 (spikes above their global P95).

Conclusion (straightforward)
- During the incident window (2024-01-20 07:52:39–08:22:39 UTC), the telemetry shows consistent anomalies across the candidate services:
  - socket-related metric spikes (socket kpi) across all five services,
  - widespread trace error_rate anomalies,
  - log.error_count flagged for all candidate services.
- error_logs.csv contains no raw error log entries.

If you want, I can (next) convert the Unix timestamps to human-readable UTC times, or show the top 20 rows per file as a compact table.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id  kpi_name    global_P95  total_points_in_window  anomaly_count  earliest_anomaly_ts  max_value_in_window
0         checkoutservice    socket  9.000000e+00                      25             25         1.705737e+09         9.000000e+00
1   recommendationservice    socket  4.000000e+00                      25             25         1.705737e+09         4.000000e+00
2   productcatalogservice    socket  3.000000e+00                      25             25         1.705737e+09         3.200000e+00
3         currencyservice    socket  3.000000e+00                      25             25         1.705737e+09         3.000000e+00
4            emailservice    socket  3.000000e+00                      25             25         1.705737e+09         3.000000e+00
5         currencyservice    diskio  0.000000e+00                      25             25         1.705737e+09         0.000000e+00
6            emailservice    diskio  0.000000e+00                      25             25         1.705737e+09         0.000000e+00
7   productcatalogservice    diskio  0.000000e+00                      25             25         1.705737e+09         0.000000e+00
8   recommendationservice    diskio  0.000000e+00                      25             25         1.705737e+09         0.000000e+00
9            emailservice       mem  4.376166e+07                      25              3         1.705738e+09         4.376310e+07
10        currencyservice       mem  4.576388e+07                      25              2         1.705738e+09         4.618738e+07
11  recommendationservice       mem  4.574459e+07                      25              2         1.705739e+09         4.574713e+07
12        checkoutservice       mem  1.201884e+07                      25              2         1.705737e+09         1.207644e+07
13  productcatalogservice       mem  1.140610e+07                      25              2         1.705738e+09         1.196892e+07
14  productcatalogservice  workload  1.490664e+02                      25              2         1.705738e+09         1.497780e+02
15        currencyservice  workload  8.147463e+01                      25              2         1.705738e+09         8.293343e+01
16        currencyservice       cpu  3.364747e+01                      25              2         1.705737e+09         3.455064e+01
17  recommendationservice  workload  1.968680e+01                      25              2         1.705738e+09         1.973000e+01
18  productcatalogservice       cpu  4.240997e+00                      25              2         1.705737e+09         4.382180e+00
19  recommendationservice       cpu  2.962376e+00                      25              2         1.705737e+09         3.167212e+00,                   cmdb_id                                   trace_name  global_P95  total_points_in_window  anomaly_count  earliest_anomaly_ts  max_value_in_window
0         checkoutservice        trace.from_checkoutservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
1         checkoutservice        trace.from_frontendservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
2         checkoutservice                   trace.from_root.error_rate         0.0                      25             25         1.705737e+09                  0.0
3         checkoutservice          trace.to_checkoutservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
4         checkoutservice          trace.to_currencyservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
5         checkoutservice             trace.to_emailservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
6         checkoutservice           trace.to_paymentservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
7         checkoutservice    trace.to_productcatalogservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
8         currencyservice        trace.from_checkoutservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
9         currencyservice        trace.from_frontendservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
10        currencyservice                   trace.from_root.error_rate         0.0                      25             25         1.705737e+09                  0.0
11           emailservice        trace.from_checkoutservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
12           emailservice                   trace.from_root.error_rate         0.0                      25             25         1.705737e+09                  0.0
13  productcatalogservice        trace.from_checkoutservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
14  productcatalogservice        trace.from_frontendservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
15  productcatalogservice  trace.from_recommendationservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
16  productcatalogservice                   trace.from_root.error_rate         0.0                      25             25         1.705737e+09                  0.0
17  recommendationservice        trace.from_frontendservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
18  recommendationservice  trace.from_recommendationservice.error_rate         0.0                      25             25         1.705737e+09                  0.0
19  recommendationservice                   trace.from_root.error_rate         0.0                      25             25         1.705737e+09                  0.0,                  cmdb_id         log_name  global_P95  total_points_in_window  anomaly_count  earliest_anomaly_ts  max_value_in_window
0        checkoutservice  log.error_count         0.0                      25             25         1.705737e+09                    0
1        currencyservice  log.error_count         0.0                      25             25         1.705737e+09                    0
2           emailservice  log.error_count         0.0                      25             25         1.705737e+09                    0
3  recommendationservice  log.error_count         0.0                      25             25         1.705737e+09                    0
4        currencyservice    log.row_count      2489.8                      25              2         1.705737e+09                 2497
5  recommendationservice    log.row_count       594.0                      25              2         1.705738e+09                  598
6        checkoutservice    log.row_count       107.4                      25              2         1.705738e+09                  118
7           emailservice    log.row_count        35.8                      25              2         1.705738e+09                   39,              file  distinct_cmdbid_kpi_groups_processed error_logs_empty
0      metric.csv                                    34             None
1       trace.csv                                    88             None
2         log.csv                                     8             None
3  error_logs.csv                                     0             True)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse mdf if present, otherwise load
try:
    mdf
except NameError:
    mdf = pd.read_csv("metric.csv")

# Candidate services
cands = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

# Ensure timestamp numeric and datetime parsed to UTC
mdf["timestamp"] = pd.to_numeric(mdf["timestamp"], errors="coerce")
mdf["ts"] = pd.to_datetime(mdf["timestamp"], unit="s", utc=True)

# Ensure value numeric
mdf["value"] = pd.to_numeric(mdf["value"], errors="coerce")

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-20 07:52:39", utc=True)
window_end = pd.to_datetime("2024-01-20 08:22:39", utc=True)

rows = []
for svc in cands:
    df_full = mdf[(mdf["cmdb_id"] == svc) & (mdf["kpi_name"] == "socket")].copy().sort_values("timestamp")
    if df_full.shape[0] == 0:
        # no data for this service/kpi
        rows.append({
            "cmdb_id": svc,
            "global_P95": np.nan,
            "number_of_anomaly_runs_in_window": 0,
            "longest_run_start_ts": np.nan,
            "longest_run_end_ts": np.nan,
            "longest_run_length": 0,
            "longest_run_max_value": np.nan,
            "severity_ratio": np.nan
        })
        continue

    # compute global P95 using full series (rule 9)
    global_P95 = df_full["value"].quantile(0.95)

    # windowed rows
    df_win = df_full[(df_full["ts"] >= window_start) & (df_full["ts"] <= window_end)].copy().sort_values("timestamp")

    if df_win.shape[0] == 0:
        rows.append({
            "cmdb_id": svc,
            "global_P95": float(global_P95),
            "number_of_anomaly_runs_in_window": 0,
            "longest_run_start_ts": np.nan,
            "longest_run_end_ts": np.nan,
            "longest_run_length": 0,
            "longest_run_max_value": np.nan,
            "severity_ratio": ( "inf" if global_P95 <= 0 else np.nan )
        })
        continue

    # detect anomaly points where value >= global_P95
    df_win["is_anom"] = (df_win["value"] >= global_P95).astype(int)

    if df_win["is_anom"].sum() == 0:
        rows.append({
            "cmdb_id": svc,
            "global_P95": float(global_P95),
            "number_of_anomaly_runs_in_window": 0,
            "longest_run_start_ts": np.nan,
            "longest_run_end_ts": np.nan,
            "longest_run_length": 0,
            "longest_run_max_value": np.nan,
            "severity_ratio": ( "inf" if global_P95 <= 0 else np.nan )
        })
        continue

    # identify consecutive runs
    df_win = df_win.reset_index(drop=True)
    # group id increments when is_anom value changes
    df_win["block"] = (df_win["is_anom"] != df_win["is_anom"].shift(1)).cumsum()
    # aggregate only blocks where is_anom == 1
    anom_blocks = df_win[df_win["is_anom"] == 1].groupby("block").agg(
        start_ts_num = ("timestamp", "first"),
        end_ts_num = ("timestamp", "last"),
        length = ("timestamp", "count"),
        max_value = ("value", "max")
    ).reset_index(drop=True)

    number_of_runs = anom_blocks.shape[0]

    # find longest run (by length). If tie, pick earliest start_ts
    if number_of_runs > 0:
        anom_blocks = anom_blocks.sort_values(by=["length", "start_ts_num"], ascending=[False, True]).reset_index(drop=True)
        longest = anom_blocks.iloc[0]
        longest_start = int(longest["start_ts_num"])
        longest_end = int(longest["end_ts_num"])
        longest_len = int(longest["length"])
        longest_max = float(longest["max_value"])
    else:
        longest_start = np.nan
        longest_end = np.nan
        longest_len = 0
        longest_max = np.nan

    # severity_ratio per instruction
    if pd.isna(global_P95):
        severity = np.nan
    else:
        if global_P95 > 0:
            severity = float(longest_max / global_P95) if not pd.isna(longest_max) else np.nan
        else:
            severity = "inf"

    rows.append({
        "cmdb_id": svc,
        "global_P95": float(global_P95),
        "number_of_anomaly_runs_in_window": int(number_of_runs),
        "longest_run_start_ts": longest_start,
        "longest_run_end_ts": longest_end,
        "longest_run_length": int(longest_len),
        "longest_run_max_value": longest_max,
        "severity_ratio": severity
    })

result_df = pd.DataFrame(rows)

# Determine summary: earliest longest_run_start_ts and largest severity_ratio
# For comparison, build numeric severity_for_sort (treat 'inf' as np.inf)
def sev_sort_val(x):
    if x == "inf":
        return np.inf
    try:
        return float(x)
    except:
        return np.nan

result_df["severity_sort"] = result_df["severity_ratio"].apply(sev_sort_val)
# earliest longest run start (exclude NaN)
earliest_row = result_df[result_df["longest_run_start_ts"].notna()].sort_values("longest_run_start_ts").head(1)
earliest_service = earliest_row["cmdb_id"].iloc[0] if not earliest_row.empty else None
earliest_ts = int(earliest_row["longest_run_start_ts"].iloc[0]) if not earliest_row.empty else None

# largest severity
largest_row = result_df[result_df["severity_sort"].notna()].sort_values("severity_sort", ascending=False).head(1)
largest_service = largest_row["cmdb_id"].iloc[0] if not largest_row.empty else None
largest_sev = largest_row["severity_ratio"].iloc[0] if not largest_row.empty else None

# Cleanup display: drop helper col
result_df = result_df[["cmdb_id", "global_P95", "number_of_anomaly_runs_in_window",
                       "longest_run_start_ts", "longest_run_end_ts", "longest_run_length",
                       "longest_run_max_value", "severity_ratio"]]

summary_line = f"earliest_longest_run_service={earliest_service}, earliest_start_ts={earliest_ts}; largest_severity_service={largest_service}, severity={largest_sev}"

# Display compact results
result_df, summary_line

```
Out[3]:
```
Summary (plain English, compact):

- For the 'socket' KPI in the five candidate services, each service had one continuous anomaly run during the incident window:
  - checkoutservice — global_P95=9.0; anomaly runs=1; longest run start=1705737300; end=1705738740; length=25 points; max_value=9.0; severity_ratio=1.00
  - currencyservice — global_P95=3.0; anomaly runs=1; longest run start=1705737300; end=1705738740; length=25; max_value=3.0; severity_ratio=1.00
  - emailservice — global_P95=3.0; anomaly runs=1; longest run start=1705737300; end=1705738740; length=25; max_value=3.0; severity_ratio=1.00
  - productcatalogservice — global_P95=3.0; anomaly runs=1; longest run start=1705737300; end=1705738740; length=25; max_value=3.2; severity_ratio≈1.0667
  - recommendationservice — global_P95=4.0; anomaly runs=1; longest run start=1705737300; end=1705738740; length=25; max_value=4.0; severity_ratio=1.00

Final summary line:
- earliest longest-run start: checkoutservice (start_ts=1705737300)
- largest severity ratio: productcatalogservice (severity_ratio ≈ 1.0667)

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id  global_P95  number_of_anomaly_runs_in_window  longest_run_start_ts  longest_run_end_ts  longest_run_length  longest_run_max_value  severity_ratio
0        checkoutservice         9.0                                 1            1705737300          1705738740                  25                    9.0        1.000000
1        currencyservice         3.0                                 1            1705737300          1705738740                  25                    3.0        1.000000
2           emailservice         3.0                                 1            1705737300          1705738740                  25                    3.0        1.000000
3  productcatalogservice         3.0                                 1            1705737300          1705738740                  25                    3.2        1.066667
4  recommendationservice         4.0                                 1            1705737300          1705738740                  25                    4.0        1.000000, 'earliest_longest_run_service=checkoutservice, earliest_start_ts=1705737300; largest_severity_service=productcatalogservice, severity=1.0666666666666667')```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import re

# Reuse tdf if present, otherwise load
try:
    tdf
except NameError:
    tdf = pd.read_csv("trace.csv")

# Candidate services
cands = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

# Ensure timestamp numeric and parsed to UTC datetimes
tdf["timestamp"] = pd.to_numeric(tdf["timestamp"], errors="coerce")
tdf["ts"] = pd.to_datetime(tdf["timestamp"], unit="s", utc=True)
tdf["value"] = pd.to_numeric(tdf["value"], errors="coerce")

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-20 07:52:39", utc=True)
window_end = pd.to_datetime("2024-01-20 08:22:39", utc=True)

# Restrict to candidate services
tdf_c = tdf[tdf["cmdb_id"].isin(cands)].copy()

# Compute global P95 per (cmdb_id, trace_name) using full series (rule 9)
gp = (
    tdf_c.groupby(["cmdb_id", "trace_name"], as_index=False)["value"]
    .quantile(0.95)
    .rename(columns={"value": "global_P95"})
)

# Merge global_P95 back into candidate trace dataframe
tdf_c = tdf_c.merge(gp, on=["cmdb_id", "trace_name"], how="left")

# Window filter
win_mask = (tdf_c["ts"] >= window_start) & (tdf_c["ts"] <= window_end)
tdf_win = tdf_c[win_mask].copy().sort_values(["cmdb_id", "trace_name", "timestamp"])

# Mark anomalies (value >= global_P95)
tdf_win["is_anom"] = (tdf_win["value"] >= tdf_win["global_P95"]).astype(int)

# Aggregate per group for windowed data
if tdf_win.shape[0] > 0:
    agg = (
        tdf_win.groupby(["cmdb_id", "trace_name"], as_index=False)
        .agg(
            total_points_in_window = ("value", "count"),
            anomaly_count_in_window = ("is_anom", "sum"),
            # earliest_anomaly_ts: min timestamp where is_anom==1, else NaN
            earliest_anomaly_ts = ("timestamp", lambda s: int(s[tdf_win.loc[s.index, "is_anom"].astype(bool)].min()) if (tdf_win.loc[s.index, "is_anom"].astype(bool).any()) else np.nan),
            max_value_in_window = ("value", "max")
        )
    )
else:
    agg = pd.DataFrame(columns=["cmdb_id", "trace_name", "total_points_in_window", "anomaly_count_in_window", "earliest_anomaly_ts", "max_value_in_window"])

# Merge with global P95 to ensure groups present and keep global_P95
summary = gp.merge(agg, on=["cmdb_id", "trace_name"], how="left")

# Fill NaNs for counts with zeros
summary["total_points_in_window"] = summary["total_points_in_window"].fillna(0).astype(int)
summary["anomaly_count_in_window"] = summary["anomaly_count_in_window"].fillna(0).astype(int)
summary["earliest_anomaly_ts"] = summary["earliest_anomaly_ts"].where(summary["earliest_anomaly_ts"].notna(), np.nan)
summary["max_value_in_window"] = summary["max_value_in_window"].where(summary["max_value_in_window"].notna(), np.nan)

# Filter groups where anomaly_count_in_window > 0
trace_anom_df = summary[summary["anomaly_count_in_window"] > 0].copy()

# Sort and limit to top 50 rows (by anomaly_count desc then max_value desc)
trace_anom_df = trace_anom_df.sort_values(by=["anomaly_count_in_window", "max_value_in_window"], ascending=[False, False]).head(50).reset_index(drop=True)

# Extract downstream target services from trace_name matching pattern 'trace.to_<servicename>.*'
def extract_target(name):
    m = re.search(r"^trace\.to_([^\.]+)", name)
    return m.group(1) if m else None

trace_anom_df["target_service"] = trace_anom_df["trace_name"].apply(extract_target)

# Build compact summary of target services: sum anomaly counts and earliest anomaly ts across any matching group
targets = (
    trace_anom_df[trace_anom_df["target_service"].notna()]
    .groupby("target_service", as_index=False)
    .agg(
        summed_anomaly_count = ("anomaly_count_in_window", "sum"),
        earliest_anomaly_ts = ("earliest_anomaly_ts", "min")
    )
)

# Ensure earliest_anomaly_ts is integer where present
targets["earliest_anomaly_ts"] = targets["earliest_anomaly_ts"].where(targets["earliest_anomaly_ts"].notna(), np.nan)

# Compact outputs: trace_anom_df (selected columns) and targets summary
trace_anom_df_out = trace_anom_df[["cmdb_id", "trace_name", "global_P95", "total_points_in_window", "anomaly_count_in_window", "earliest_anomaly_ts", "max_value_in_window", "target_service"]]
targets_out = targets.rename(columns={"target_service": "target_service", "summed_anomaly_count": "summed_anomaly_count", "earliest_anomaly_ts": "earliest_anomaly_ts"})

# Display results (limited)
trace_anom_df_out, targets_out

```
Out[4]:
```
Summary (plain English, compact):

- I examined trace.csv for the five candidate services and returned the top 50 (cmdb_id, trace_name) groups with anomaly_count_in_window > 0 for the incident window 2024-01-20 07:52:39–08:22:39 UTC. Many trace.error_rate series and some duration metrics were flagged.

- Observations from the returned groups (top examples):
  - Numerous trace.*.error_rate groups for checkoutservice, currencyservice, emailservice, productcatalogservice and recommendationservice show global_P95 = 0 and anomaly_count_in_window = 25 (i.e., all 25 points in the window met value >= global_P95).
  - Several duration-related trace series (e.g., duration_mean, duration_p95) show smaller anomaly counts (typically 2) with nonzero global_P95 and max values > 0.

- Extracted downstream target services from trace_name pattern "trace.to_<servicename>.*" and aggregated anomalies:
  - productcatalogservice — summed_anomaly_count = 54, earliest_anomaly_ts = 1705737300
  - checkoutservice — summed_anomaly_count = 29, earliest_anomaly_ts = 1705737300
  - currencyservice — summed_anomaly_count = 28, earliest_anomaly_ts = 1705737300
  - emailservice — summed_anomaly_count = 27, earliest_anomaly_ts = 1705737300
  - paymentservice — summed_anomaly_count = 27, earliest_anomaly_ts = 1705737300
  - recommendationservice — summed_anomaly_count = 27, earliest_anomaly_ts = 1705737300

- Compact conclusion:
  - Trace anomalies are widespread across the candidate services during the incident window. productcatalogservice stands out with the largest aggregated anomaly count (54) among downstream targets. All earliest anomaly timestamps reported for these targets are 1705737300 (Unix seconds).

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                trace_name  global_P95  total_points_in_window  anomaly_count_in_window  earliest_anomaly_ts  max_value_in_window   target_service
0   checkoutservice     trace.from_checkoutservice.error_rate    0.000000                      25                       25           1705737300             0.000000             None
1   checkoutservice     trace.from_frontendservice.error_rate    0.000000                      25                       25           1705737300             0.000000             None
2   checkoutservice                trace.from_root.error_rate    0.000000                      25                       25           1705737300             0.000000             None
3   checkoutservice       trace.to_checkoutservice.error_rate    0.000000                      25                       25           1705737300             0.000000  checkoutservice
4   checkoutservice       trace.to_currencyservice.error_rate    0.000000                      25                       25           1705737300             0.000000  currencyservice
..              ...                                       ...         ...                     ...                      ...                  ...                  ...              ...
45  checkoutservice  trace.from_frontendservice.duration_mean    0.307557                      25                        2           1705738200             0.315064             None
46  checkoutservice   trace.from_checkoutservice.duration_p95    0.204345                      25                        2           1705738080             0.204352             None
47  checkoutservice     trace.to_checkoutservice.duration_p95    0.204345                      25                        2           1705738080             0.204352  checkoutservice
48  checkoutservice              trace.from_root.duration_p95    0.072510                      25                        2           1705737720             0.073328             None
49  checkoutservice  trace.from_checkoutservice.duration_mean    0.028731                      25                        2           1705738380             0.030761             None

[50 rows x 8 columns],           target_service  summed_anomaly_count  earliest_anomaly_ts
0        checkoutservice                    29           1705737300
1        currencyservice                    28           1705737300
2           emailservice                    27           1705737300
3         paymentservice                    27           1705737300
4  productcatalogservice                    54           1705737300
5  recommendationservice                    27           1705737300)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import re

# Reuse tdf if present; otherwise load trace.csv
try:
    tdf
except NameError:
    tdf = pd.read_csv("trace.csv")

# Candidate services
cands = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

# Prepare timestamps and numeric values (respecting UTC)
tdf["timestamp"] = pd.to_numeric(tdf["timestamp"], errors="coerce")
tdf["ts"] = pd.to_datetime(tdf["timestamp"], unit="s", utc=True)
tdf["value"] = pd.to_numeric(tdf["value"], errors="coerce")

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-20 07:52:39", utc=True)
window_end = pd.to_datetime("2024-01-20 08:22:39", utc=True)

# Restrict to candidate services
tdf_c = tdf[tdf["cmdb_id"].isin(cands)].copy()

# Compute global P95 per (cmdb_id, trace_name) using full series
gp = (
    tdf_c.groupby(["cmdb_id", "trace_name"], as_index=False)["value"]
    .quantile(0.95)
    .rename(columns={"value": "global_P95"})
)

# Merge global_P95 back
tdf_c = tdf_c.merge(gp, on=["cmdb_id", "trace_name"], how="left")

# Window-filtered rows
win_mask = (tdf_c["ts"] >= window_start) & (tdf_c["ts"] <= window_end)
tdf_win = tdf_c[win_mask].copy().sort_values(["cmdb_id", "trace_name", "timestamp"])

# Mark anomalies in window (value >= global_P95)
tdf_win["is_anom"] = (tdf_win["value"] >= tdf_win["global_P95"]).astype(int)

# Aggregate window stats per group
if not tdf_win.empty:
    agg_counts = (
        tdf_win.groupby(["cmdb_id", "trace_name"], as_index=False)
        .agg(
            total_points_in_window = ("value", "count"),
            anomaly_count_in_window = ("is_anom", "sum"),
            max_value_in_window = ("value", "max")
        )
    )
    # earliest anomaly timestamp per group (only where is_anom==1)
    earliest = (
        tdf_win[tdf_win["is_anom"] == 1]
        .groupby(["cmdb_id", "trace_name"], as_index=False)["timestamp"]
        .min()
        .rename(columns={"timestamp": "earliest_anomaly_ts"})
    )
    agg = agg_counts.merge(earliest, on=["cmdb_id", "trace_name"], how="left")
else:
    agg = pd.DataFrame(columns=["cmdb_id", "trace_name", "total_points_in_window",
                                "anomaly_count_in_window", "max_value_in_window", "earliest_anomaly_ts"])

# Merge with global P95 to include that column
trace_summary = gp.merge(agg, on=["cmdb_id", "trace_name"], how="left")

# Fill zeros for counts where appropriate
trace_summary["total_points_in_window"] = trace_summary["total_points_in_window"].fillna(0).astype(int)
trace_summary["anomaly_count_in_window"] = trace_summary["anomaly_count_in_window"].fillna(0).astype(int)
trace_summary["earliest_anomaly_ts"] = trace_summary["earliest_anomaly_ts"].where(trace_summary["earliest_anomaly_ts"].notna(), np.nan)
trace_summary["max_value_in_window"] = trace_summary["max_value_in_window"].where(trace_summary["max_value_in_window"].notna(), np.nan)

# Filter to groups with anomaly_count_in_window > 0 and trace_name starting with 'trace.to_'
trace_to_anoms = trace_summary[
    (trace_summary["anomaly_count_in_window"] > 0) &
    (trace_summary["trace_name"].str.startswith("trace.to_"))
].copy()

# Parse target_service from trace_name pattern 'trace.to_<servicename>(.|$)'
trace_to_anoms["target_service"] = trace_to_anoms["trace_name"].str.extract(r"^trace\.to_([^\.]+)", expand=False)

# Aggregate by origin_service (cmdb_id) and target_service
origin_target_agg = (
    trace_to_anoms.groupby(["cmdb_id", "target_service"], as_index=False)
    .agg(
        summed_anomaly_count = ("anomaly_count_in_window", "sum"),
        earliest_anomaly_ts = ("earliest_anomaly_ts", "min")
    )
    .rename(columns={"cmdb_id": "origin_service"})
)

# Ensure earliest_anomaly_ts is integer where present
origin_target_agg["earliest_anomaly_ts"] = origin_target_agg["earliest_anomaly_ts"].where(origin_target_agg["earliest_anomaly_ts"].notna(), np.nan)

# Sort by summed_anomaly_count desc and limit to top 20
origin_target_top = origin_target_agg.sort_values(by=["summed_anomaly_count", "earliest_anomaly_ts"], ascending=[False, True]).head(20).reset_index(drop=True)

# Summary line: which target_service has largest summed_anomaly_count (across origins) and its earliest_anomaly_ts
# First aggregate across target_service only
target_totals = (
    origin_target_agg.groupby("target_service", as_index=False)
    .agg(summed_anomaly_count = ("summed_anomaly_count", "sum"),
         earliest_anomaly_ts = ("earliest_anomaly_ts", "min"))
    .sort_values(by="summed_anomaly_count", ascending=False)
    .reset_index(drop=True)
)

if not target_totals.empty:
    top_target = target_totals.iloc[0]
    summary_line = f"top_target_service={top_target['target_service']}, summed_anomaly_count={int(top_target['summed_anomaly_count'])}, earliest_anomaly_ts={int(top_target['earliest_anomaly_ts']) if pd.notna(top_target['earliest_anomaly_ts']) else None}"
else:
    summary_line = "no target services with anomalies found"

# Display compact results
origin_target_top, summary_line

```
Out[5]:
```
Summary (plain English, compact):

I aggregated trace anomalies of pattern "trace.to_<target>.*" for the five candidate origin services during the incident window (2024-01-20 07:52:39–08:22:39 UTC). Top origin→target aggregates (summed anomaly_count, earliest_anomaly_ts):

- checkoutservice → currencyservice: summed_anomaly_count=32, earliest_anomaly_ts=1705737300  
- checkoutservice → checkoutservice: summed_anomaly_count=31, earliest_anomaly_ts=1705737300  
- checkoutservice → emailservice: summed_anomaly_count=31, earliest_anomaly_ts=1705737300  
- checkoutservice → paymentservice: summed_anomaly_count=31, earliest_anomaly_ts=1705737300  
- checkoutservice → productcatalogservice: summed_anomaly_count=31, earliest_anomaly_ts=1705737300  
- recommendationservice → productcatalogservice: summed_anomaly_count=31, earliest_anomaly_ts=1705737300  
- recommendationservice → recommendationservice: summed_anomaly_count=31, earliest_anomaly_ts=1705737300

Overall top target service:
- productcatalogservice — largest summed_anomaly_count = 62 (earliest_anomaly_ts = 1705737300).

Interpretation:
- Many origin services (particularly checkoutservice and recommendationservice) show elevated outbound trace anomalies targeting multiple services. productcatalogservice stands out as the most affected downstream target (highest total anomaly count across origins). All earliest anomaly timestamps reported are 1705737300.

The original code execution output of IPython Kernel is also provided below for reference:

(          origin_service         target_service  summed_anomaly_count  earliest_anomaly_ts
0        checkoutservice        currencyservice                    32           1705737300
1        checkoutservice        checkoutservice                    31           1705737300
2        checkoutservice           emailservice                    31           1705737300
3        checkoutservice         paymentservice                    31           1705737300
4        checkoutservice  productcatalogservice                    31           1705737300
5  recommendationservice  productcatalogservice                    31           1705737300
6  recommendationservice  recommendationservice                    31           1705737300, 'top_target_service=productcatalogservice, summed_anomaly_count=62, earliest_anomaly_ts=1705737300')```
```